# A FAIRE POUR LANCER UN GROS ENTRAINEMENT

- utiliser un modèle b3 (ou +) que b0
- en csq changer le target size dans le preprocessor
- augmenter nombre d'epoch
- batch size ?

Optionnel :
- tester une autre loss

# **0. Librairies**

In [1]:
import torch
import torch.nn as nn
from torchvision import models
from torch.utils.data import DataLoader
from tqdm import tqdm
import sys
import os




sys.path.append(os.path.abspath(".."))

from lib.data.dataset import BeeDataset

from lib.data.preprocessing import TorchPreprocessor

from lib.data.train_val_split import train_val_split

from lib.data.preprocessing import TorchPreprocessor

from lib.data.data_augmentation import data_augmented_loader

In [2]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

BATCH_SIZE = 4
EPOCHS = 10
LR = 1e-3


notebook_dir = os.getcwd()

data_dir = os.path.abspath(os.path.join(notebook_dir, "..", "data"))





## **1. Preprocessing**

In [3]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

# preprocessor = TorchPreprocessor(
#     normalize=True,
#     mean = IMAGENET_MEAN,
#     std = IMAGENET_STD,
#     resize_method="pad",
#     target_size=(224, 224),
# )

# train_dataset, val_dataset = train_val_split(train_transform=preprocessor, val_transform=preprocessor)


In [4]:
import lib.data.preprocessing as prep
print(prep.__file__)

/home/audrey-maurette/Documents/ISAE-Supaero/SDD/Hackaton_abeilles_tigres/lib/data/preprocessing.py


In [5]:
train_loader, val_loader = data_augmented_loader(IMAGENET_MEAN, IMAGENET_STD)

Train prêt : 6417 images (avec augmentation)
Val prête  : 1582 images (sans augmentation)


In [6]:
# Récupérer le dataset utilisé dans le DataLoader
train_dataset = train_loader.dataset

# Extraire tous les labels uniques
labels = [sample[1] for sample in train_dataset.samples]

# Nombre de classes
num_classes = len(set(labels))

print(f"Nombre de classes: {num_classes}")

print("Nombre de classes :", num_classes)


Nombre de classes: 50
Nombre de classes : 50


## **2. Modèle**

In [7]:
def create_model(num_classes: int) -> nn.Module:
    model = models.efficientnet_b0(weights="IMAGENET1K_V1") #mettre b3 si ca marche
    in_features = model.classifier[1].in_features
    model.classifier = nn.Sequential(
        nn.Dropout(p=0.4),
        nn.Linear(in_features, num_classes),
    )
    return model

model = create_model(num_classes)
model.to(DEVICE)



criterion = nn.CrossEntropyLoss()

# --- Variante ---
# pip install torchmetrics
# from torchmetrics.classification import MulticlassFocalLoss
# criterion = MulticlassFocalLoss(num_classes=num_classes, alpha=0.25, gamma=2.0)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LR,
    weight_decay=1e-4
)


## **3. F1-score**

In [8]:
import numpy as np

def compute_f1(all_labels, all_preds, num_classes):
    f1_per_class = []

    for cls in range(num_classes):
        # True Positives
        TP = np.sum((np.array(all_preds) == cls) & (np.array(all_labels) == cls))
        # False Positives
        FP = np.sum((np.array(all_preds) == cls) & (np.array(all_labels) != cls))
        # False Negatives
        FN = np.sum((np.array(all_preds) != cls) & (np.array(all_labels) == cls))

        precision = TP / (TP + FP) if (TP + FP) > 0 else 0
        recall    = TP / (TP + FN) if (TP + FN) > 0 else 0

        f1 = (2 * precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
        f1_per_class.append(f1)

    # F1 macro : moyenne des classes
    f1_macro = np.mean(f1_per_class)
    return f1_macro, f1_per_class

## **4. Fonctions de training et validation**

In [9]:
def train_one_epoch():
    model.train()
    total_loss = 0
    correct = 0
    total = 0

    all_preds = []
    all_labels = []

    for images, labels in tqdm(train_loader):
        images, labels = images.to(DEVICE), labels.to(DEVICE)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        preds = outputs.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

    # 🔹 Calcul F1 avec ta fonction existante
    f1_macro, f1_per_class = compute_f1(all_labels, all_preds, num_classes)

    # 🔹 Affichage
    # print(f"Train F1 macro: {f1_macro:.4f}")

    return total_loss / len(train_loader), correct / total, f1_macro, f1_per_class


In [10]:
import torch

def validate():
    model.eval()
    total_loss = 0

    all_preds = []
    all_labels = []

    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            
            outputs = model(images)
            loss = criterion(outputs, labels)
            total_loss += loss.item()
            
            preds = outputs.argmax(dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    
    # Calcul F1 manuel par classe
    f1_per_class = []
    for cls in range(num_classes):
        TP = np.sum((np.array(all_preds) == cls) & (np.array(all_labels) == cls))
        FP = np.sum((np.array(all_preds) == cls) & (np.array(all_labels) != cls))
        FN = np.sum((np.array(all_preds) != cls) & (np.array(all_labels) == cls))
        
        precision = TP / (TP + FP) if (TP + FP) > 0 else 0
        recall    = TP / (TP + FN) if (TP + FN) > 0 else 0
        
        f1 = (2 * precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
        f1_per_class.append(f1)
    
    f1_macro = np.mean(f1_per_class)

    return total_loss / len(val_loader), f1_macro, f1_per_class

## **5. Entrainement**

**Vérif des labels**

In [11]:
all_labels = [label for _, label in train_dataset.samples]
print("Label min:", min(all_labels))
print("Label max:", max(all_labels))
print("Nombre de classes uniques:", len(set(all_labels)))

# Récupérer tous les labels uniques triés
all_labels = sorted(set(label for _, label in train_dataset.samples))
label_to_index = {label: i for i, label in enumerate(all_labels)}

# Remapper les labels dans le dataset
# for i, (path, label) in enumerate(train_dataset.samples):
#     train_dataset.samples[i] = (path, label_to_index[label])

Label min: 0
Label max: 49
Nombre de classes uniques: 50


**Entrainement**

In [12]:
best_f1 = 0.0
best_model_path = "best_model.pth"

for epoch in range(EPOCHS):
    train_loss, train_acc, train_f1_macro, train_f1_per_class = train_one_epoch()
    print(f"\nEpoch {epoch+1}/{EPOCHS}")
    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
    print(f"Train F1 Macro: {train_f1_macro:.4f}")
    print(f"Train F1 per class: {train_f1_per_class}")

    val_loss, val_f1_macro, val_f1_per_class = validate()
    print(f"Val   Loss: {val_loss:.4f}")
    print(f"Val   F1 Macro: {val_f1_macro:.4f}")
    print(f"Val   F1 per class: {val_f1_per_class}")

    # 🔹 Sauvegarde du meilleur modèle
    if val_f1_macro > best_f1:
        best_f1 = val_f1_macro
        torch.save(model.state_dict(), best_model_path)
        print(f" Nouveau meilleur modèle sauvegardé ! F1 Macro = {best_f1:.4f}")

  0%|          | 0/201 [00:00<?, ?it/s]

100%|██████████| 201/201 [09:30<00:00,  2.84s/it]


Epoch 1/10
Train Loss: 1.8134 | Train Acc: 0.5306
Train F1 Macro: 0.5180
Train F1 per class: [np.float64(0.803030303030303), np.float64(0.783625730994152), np.float64(0.1581920903954802), np.float64(0.721518987341772), np.float64(0.5575221238938054), np.float64(0.34871794871794876), np.float64(0.5075757575757575), np.float64(0.7119741100323624), np.float64(0.7586206896551724), np.float64(0.45614035087719296), np.float64(0.7159533073929961), np.float64(0.62015503875969), np.float64(0.19696969696969693), np.float64(0.6832740213523132), np.float64(0.4046692607003891), np.float64(0.3681592039800995), np.float64(0.5291828793774318), np.float64(0.2559241706161137), np.float64(0.3488372093023256), np.float64(0.5485232067510549), np.float64(0.7623762376237623), np.float64(0.415929203539823), np.float64(0.40689655172413786), np.float64(0.5670498084291187), np.float64(0.28193832599118945), np.float64(0.2730627306273063), np.float64(0.7756653992395437), np.float64(0.672), np.float64(0.1433962264

Val   Loss: 1.8354
Val   F1 Macro: 0.3727
Val   F1 per class: [0, np.float64(1.0), np.float64(0.26666666666666666), np.float64(0.5), np.float64(0.5714285714285715), np.float64(0.37168141592920356), np.float64(0.36363636363636365), 0, np.float64(0.5), np.float64(0.33333333333333337), np.float64(1.0), np.float64(0.4), np.float64(0.32812499999999994), 0, np.float64(0.358974358974359), np.float64(0.4), np.float64(0.18181818181818182), np.float64(0.32), np.float64(0.5658536585365854), np.float64(0.7032967032967032), 0, np.float64(0.21428571428571427), np.float64(0.3780487804878048), np.float64(0.7843137254901961), np.float64(0.04724409448818897), np.float64(0.6642857142857141), np.float64(1.0), 0, np.float64(0.3857868020304568), np.float64(0.07142857142857144), 0, np.float64(0.5), np.float64(0.14285714285714285), np.float64(0.4), np.float64(1.0), 0, np.float64(0.4426229508196721), np.float64(0.37037037037037035), 0, np.float64(0.2857142857142857), np.float64(0.5911602209944751), np.float64(

100%|██████████| 201/201 [09:52<00:00,  2.95s/it]

...

100%|██████████| 201/201 [09:49<00:00,  2.93s/it]


Epoch 3/10
Train Loss: 0.8519 | Train Acc: 0.7556
Train F1 Macro: 0.7521
Train F1 per class: [np.float64(0.9541984732824427), np.float64(0.9721115537848605), np.float64(0.4206896551724138), np.float64(0.8827586206896553), np.float64(0.8660714285714286), np.float64(0.5758754863813229), np.float64(0.8442906574394463), np.float64(0.9155555555555556), np.float64(0.9076305220883534), np.float64(0.7410358565737052), np.float64(0.9435215946843853), np.float64(0.8058608058608059), np.float64(0.38999999999999996), np.float64(0.890625), np.float64(0.6569343065693432), np.float64(0.7520000000000001), np.float64(0.8273092369477912), np.float64(0.5742574257425742), np.float64(0.5), np.float64(0.7942238267148015), np.float64(0.9477351916376306), np.float64(0.7751937984496123), np.float64(0.616600790513834), np.float64(0.7330960854092526), np.float64(0.4200913242009133), np.float64(0.6591760299625469), np.float64(0.9147982062780269), np.float64(0.9225806451612903), np.float64(0.4860557768924303), np

Val   Loss: 1.4812
Val   F1 Macro: 0.4599
Val   F1 per class: [0, np.float64(1.0), np.float64(0.4675324675324675), np.float64(0.6666666666666666), 0, np.float64(0.5378151260504201), np.float64(0.4444444444444445), np.float64(1.0), np.float64(0.6666666666666666), np.float64(0.28571428571428575), np.float64(1.0), np.float64(0.2727272727272727), np.float64(0.46846846846846846), 0, np.float64(0.5384615384615384), np.float64(0.27027027027027023), np.float64(0.4), np.float64(0.3709677419354839), np.float64(0.6091954022988506), np.float64(0.8837209302325583), 0, np.float64(0.42857142857142855), np.float64(0.5917159763313609), np.float64(0.8070175438596492), np.float64(0.4835164835164835), np.float64(0.6504065040650406), np.float64(0.6666666666666666), np.float64(0.2857142857142857), np.float64(0.378698224852071), np.float64(0.6666666666666666), 0, np.float64(0.6666666666666666), np.float64(0.5245901639344263), np.float64(1.0), np.float64(0.4), 0, np.float64(0.4216867469879518), np.float64(0.4

 25%|██▍       | 50/201 [02:33<07:42,  3.06s/it]


KeyboardInterrupt: 

## **6. Création du fichier submission**

In [29]:
from torch.utils.data import DataLoader
import pandas as pd
import torch
from lib.data.dataset import BeeDataset

def submission(model, batch_size=32, transform=None, model_path="best_model.pth", save_path="submission.csv"):
    # Charger le modèle sur le bon device
    model.load_state_dict(torch.load(model_path, map_location=torch.device(DEVICE)))
    model.to(DEVICE)
    model.eval()
    
    # Dataset de test
    test_dataset = BeeDataset(train=False, transform=transform)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)
    
    all_ids = []
    all_preds = []
    
    with torch.no_grad():
        for imgs, ids in test_loader:
            imgs = imgs.to(DEVICE)
            outputs = model(imgs)
            
            preds = torch.argmax(outputs, dim=1)
            
            # Convertir preds en int et ids en int ou str
            all_preds.extend(preds.cpu().tolist())
            all_ids.extend([int(x) if isinstance(x, torch.Tensor) else x for x in ids])
    
    submission_df = pd.DataFrame({
        "id": all_ids,
        "label": all_preds
    })
    
    submission_df.to_csv(save_path, index=False)
    print(f"Submission saved to {save_path}")

In [30]:
preprocessor = TorchPreprocessor(
    normalize=True,
    mean = IMAGENET_MEAN,
    std = IMAGENET_STD,
    resize_method="pad",
    target_size=(224, 224),
)


test_dataset = BeeDataset(train=False, transform=preprocessor)


submission(model, batch_size=32, transform=preprocessor, save_path="submission.csv")

Submission saved to submission.csv
